# Deep Learning NLP Text Classifier — Spam Filter (PyTorch)

Builds a small feed-forward neural network in **PyTorch** that classifies text messages
as spam or ham (not spam): TF-IDF vectorization, a multi-layer network with Dropout and
Batch Normalization, a training loop with manual early stopping, convergence-curve
plots, and inference on brand-new example messages with confidence scores.

**Dataset:** a synthetic, template-generated spam/ham message set (no real dataset was
supplied for this exercise, and downloading one requires internet access this notebook
doesn't assume you have). Swap section 2 for a real dataset (e.g. the SMS Spam
Collection or a CSV of your own labeled text) — nothing downstream needs to change as
long as you end up with a `texts` list and a `labels` list of 0/1.

**On vectorization:** this notebook uses TF-IDF because it needs no internet access and
no pretrained weights to download, so it always runs. If you have internet access and
want richer embeddings instead, section 3 has a commented-out Hugging Face
`sentence-transformers` alternative you can switch to — the rest of the pipeline
(the PyTorch model, training loop, and plots) works unchanged either way, since it just
consumes whatever fixed-length numeric vectors come out of this step.


## 1. Setup

If PyTorch isn't installed yet, run this in a terminal first:
`pip install torch` (see pytorch.org for the exact command for your system/GPU).


In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


## 2. Load data (synthetic stand-in)

Generates spam-like and ham-like (normal) messages from templates with randomized
slot-filling, so there's real lexical variety for TF-IDF to pick up on. Replace this
cell with your own `texts` (list of strings) and `labels` (list of 0/1) to use a real
dataset — everything below only depends on those two variables.


In [ ]:
spam_subjects = ["You", "Congratulations", "Winner", "Customer", "Dear friend", "Attention"]
spam_actions = [
    "you have WON a free {prize}! Click here to claim now",
    "act now to claim your {prize} before it expires",
    "urgent: your account will be suspended, verify immediately",
    "limited time offer: get {prize} for FREE, click the link",
    "you are pre-approved for a $1000 cash prize, claim {prize} today",
    "congratulations! you've been selected to receive a {prize}",
    "exclusive deal just for you: 90% off {prize}, act now",
    "your loan has been approved, click to receive cash instantly",
    "final notice: claim your free {prize} before midnight",
    "hot singles in your area want to chat with you now",
]
prizes = ["iPhone", "vacation", "gift card", "cash prize", "cruise", "laptop", "voucher"]

ham_subjects = ["Hey", "Hi", "Hello", "Good morning", "Quick question"]
ham_actions = [
    "are we still on for lunch {time}",
    "can you send me the report before the {time} meeting",
    "thanks for your help with the project yesterday",
    "let's catch up over coffee {time}",
    "just checking in to see how you're doing",
    "the meeting has been moved to {time}, see you then",
    "I finished the assignment, let me know your thoughts",
    "don't forget to pick up groceries on your way home",
    "happy birthday! hope you have a wonderful day",
    "can we reschedule our call to {time}",
]
times = ["tomorrow", "next week", "this afternoon", "on Friday", "at 3pm", "next Monday"]

def make_spam():
    return f"{random.choice(spam_subjects)}, {random.choice(spam_actions).format(prize=random.choice(prizes))}"

def make_ham():
    return f"{random.choice(ham_subjects)}, {random.choice(ham_actions).format(time=random.choice(times))}"

n_per_class = 1000
texts = [make_spam() for _ in range(n_per_class)] + [make_ham() for _ in range(n_per_class)]
labels = [1] * n_per_class + [0] * n_per_class  # 1 = spam, 0 = ham

combined = list(zip(texts, labels))
random.shuffle(combined)
texts, labels = zip(*combined)
texts, labels = list(texts), list(labels)

print("Total messages:", len(texts))
print("Spam rate:", round(sum(labels) / len(labels), 3))
print("\nSample messages:")
for t, l in list(zip(texts, labels))[:5]:
    print(f"  [{'SPAM' if l else 'HAM '}] {t}")


## 3. Train/validation/test split, then vectorize

Split **before** fitting the vectorizer, and fit it only on the training text — the
validation and test sets are only ever `.transform()`-ed, never used to fit the
vectorizer, to keep this leak-free the same way the earlier notebooks in this project are.


In [ ]:
X_temp, X_test, y_temp, y_test = train_test_split(
    texts, labels, test_size=0.15, random_state=RANDOM_STATE, stratify=labels
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.15, random_state=RANDOM_STATE, stratify=y_temp
)
print("Train:", len(X_train), " Val:", len(X_val), " Test:", len(X_test))

vectorizer = TfidfVectorizer(max_features=500, ngram_range=(1, 2))
X_train_vec = vectorizer.fit_transform(X_train).toarray().astype(np.float32)
X_val_vec = vectorizer.transform(X_val).toarray().astype(np.float32)
X_test_vec = vectorizer.transform(X_test).toarray().astype(np.float32)

# --- Alternative: Hugging Face sentence embeddings instead of TF-IDF (needs internet
# access to download the pretrained model the first time you run it) ---
# from sentence_transformers import SentenceTransformer
# embedder = SentenceTransformer("all-MiniLM-L6-v2")
# X_train_vec = embedder.encode(X_train).astype(np.float32)
# X_val_vec = embedder.encode(X_val).astype(np.float32)
# X_test_vec = embedder.encode(X_test).astype(np.float32)

input_dim = X_train_vec.shape[1]
print("Feature dimension:", input_dim)


## 4. PyTorch datasets and dataloaders

In [ ]:
def to_loader(X, y, batch_size=32, shuffle=False, drop_last=False):
    X_t = torch.tensor(X, dtype=torch.float32)
    y_t = torch.tensor(y, dtype=torch.float32).unsqueeze(1)
    return DataLoader(TensorDataset(X_t, y_t), batch_size=batch_size, shuffle=shuffle, drop_last=drop_last)

# drop_last=True on the training loader avoids a rare edge case where the final batch
# has only 1 sample, which BatchNorm1d can't compute batch statistics from in train mode.
train_loader = to_loader(X_train_vec, y_train, batch_size=32, shuffle=True, drop_last=True)
val_loader = to_loader(X_val_vec, y_val, batch_size=64, shuffle=False)
test_loader = to_loader(X_test_vec, y_test, batch_size=64, shuffle=False)


## 5. Model: multi-layer network with Dropout + Batch Normalization

Two hidden layers, each `Linear -> BatchNorm1d -> ReLU -> Dropout`, then a single output
logit (used with `BCEWithLogitsLoss`, which applies the sigmoid internally for numerical
stability).


In [ ]:
class TextClassifier(nn.Module):
    def __init__(self, input_dim, hidden1=128, hidden2=64, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden1),
            nn.BatchNorm1d(hidden1),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden1, hidden2),
            nn.BatchNorm1d(hidden2),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden2, 1),  # single logit; BCEWithLogitsLoss applies sigmoid
        )

    def forward(self, x):
        return self.net(x)

model = TextClassifier(input_dim).to(device)
model


## 6. Training loop with early stopping

PyTorch has no built-in "early stopping callback" the way Keras does, so this loop
implements the equivalent by hand: after every epoch, check validation loss; if it
hasn't improved for `patience` epochs in a row, stop training and reload the best
weights seen so far.


In [ ]:
def run_epoch(loader, model, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss, correct, total = 0.0, 0, 0
    with torch.set_grad_enabled(is_train):
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * xb.size(0)
            preds = (torch.sigmoid(logits) >= 0.5).float()
            correct += (preds == yb).sum().item()
            total += xb.size(0)

    return total_loss / total, correct / total


In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

max_epochs = 60
patience = 6

history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best_val_loss = float("inf")
best_state = None
epochs_without_improvement = 0

for epoch in range(1, max_epochs + 1):
    train_loss, train_acc = run_epoch(train_loader, model, criterion, optimizer)
    val_loss, val_acc = run_epoch(val_loader, model, criterion, optimizer=None)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    improved = val_loss < best_val_loss - 1e-4
    if improved:
        best_val_loss = val_loss
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    print(
        f"Epoch {epoch:2d} | train_loss={train_loss:.4f} train_acc={train_acc:.3f} "
        f"| val_loss={val_loss:.4f} val_acc={val_acc:.3f}"
        + (" *" if improved else "")
    )

    if epochs_without_improvement >= patience:
        print(f"\nEarly stopping at epoch {epoch} (no val_loss improvement for {patience} epochs).")
        break

model.load_state_dict(best_state)
print("\nRestored best weights (val_loss = %.4f)" % best_val_loss)


## 7. Convergence curves

Loss and accuracy for both training and validation sets, epoch by epoch — the required
"training curves" deliverable. A healthy fit shows both curves declining/rising together;
a widening gap between train and validation is a sign of overfitting, which is exactly
what the Dropout and early stopping above are there to limit.


In [ ]:
epochs_ran = range(1, len(history["train_loss"]) + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].plot(epochs_ran, history["train_loss"], label="Train loss")
axes[0].plot(epochs_ran, history["val_loss"], label="Val loss")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
axes[0].set_title("Loss convergence")
axes[0].legend()

axes[1].plot(epochs_ran, history["train_acc"], label="Train accuracy")
axes[1].plot(epochs_ran, history["val_acc"], label="Val accuracy")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy")
axes[1].set_title("Accuracy convergence")
axes[1].legend()

plt.tight_layout()
plt.show()


## 8. Test-set evaluation

In [ ]:
test_loss, test_acc = run_epoch(test_loader, model, criterion, optimizer=None)
print(f"Test loss: {test_loss:.4f}   Test accuracy: {test_acc:.3f}")

model.eval()
with torch.no_grad():
    test_logits = model(torch.tensor(X_test_vec, dtype=torch.float32).to(device))
    test_preds = (torch.sigmoid(test_logits) >= 0.5).float().cpu().numpy().ravel().astype(int)

print()
print(classification_report(y_test, test_preds, target_names=["ham", "spam"]))


## 9. Inference on unseen sample inputs

Brand-new sentences the model has never seen during training, run through the *same*
fitted TF-IDF vectorizer, with the predicted class and confidence (the sigmoid
probability) printed for each.


In [ ]:
new_messages = [
    "Congratulations! You have been chosen to win a brand new car, click now to claim",
    "Hey, can you send me the slides from yesterday's meeting?",
    "URGENT: verify your bank account now or it will be suspended within 24 hours",
    "Let's grab lunch on Thursday, I'm free after 12",
    "Free entry to win a luxury vacation, text WIN to 12345 now!",
    "Thanks for the birthday wishes, means a lot!",
]

new_vec = vectorizer.transform(new_messages).toarray().astype(np.float32)
model.eval()
with torch.no_grad():
    logits = model(torch.tensor(new_vec, dtype=torch.float32).to(device))
    probs = torch.sigmoid(logits).cpu().numpy().ravel()

for msg, p in zip(new_messages, probs):
    label = "SPAM" if p >= 0.5 else "HAM"
    confidence = p if p >= 0.5 else 1 - p
    print(f"[{label:4s}] (confidence {confidence:.1%})  {msg}")


## 10. Summary

- TF-IDF (1- and 2-grams, 500 features) turns each message into a fixed-length numeric
  vector, fit only on the training split.
- The model is a 2-hidden-layer feed-forward network, each layer using
  `Linear -> BatchNorm1d -> ReLU -> Dropout(0.3)`, trained with `BCEWithLogitsLoss` and Adam.
- Early stopping (patience-based, monitoring validation loss) stops training once the
  model stops improving and restores the best-seen weights, rather than just training
  for a fixed number of epochs.
- Convergence curves (section 7) and test-set metrics (section 8) are the required
  "training curves" proof; section 9 is the required "sample inference outputs" proof.
- **Caveat:** this dataset is template-generated and easy to separate, useful for
  exercising the full pipeline correctly rather than as a benchmark. Swap in a real
  labeled text dataset (section 2) for a meaningful accuracy number.
